# AI Explainer - Colab Workflow (Parler-TTS + Whisper + SDXL)

This notebook runs the entire middle pipeline on Google Colab:
1. **Voice Generation:** Uses Parler-TTS (Prompt-based Text-to-Speech)
2. **Captioning:** Uses faster-whisper for word-level timestamps
3. **Matching:** Matches script segments against your local `clip_index.json`
4. **Images:** Uses Stable Diffusion XL to generate fallbacks for missing clips.

In [ ]:
import os
from pathlib import Path

INPUT_DIR = Path('/content/inputs')
AUDIO_DIR = Path('/content/audio')
CAPTIONS_DIR = Path('/content/captions')
IMAGES_DIR = Path('/content/images')
OUTPUT_DIR = Path('/content/output')

for d in [INPUT_DIR, AUDIO_DIR, CAPTIONS_DIR, IMAGES_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories created!\n')
print('ACTION REQUIRED: Please open the files tab on the left and upload your:')
print('1. script.txt to /content/inputs/')
print('2. clip_index.json to /content/inputs/')
print('3. (Optional) voice_prompt.txt to /content/inputs/ if you want to customize the narrator\'s voice style.')

In [ ]:
# 1. Install Dependencies
!pip install -q faster-whisper diffusers transformers accelerate safetensors huggingface_hub soundfile

# Install Parler-TTS directly from GitHub to get the latest stable version
!pip install -q git+https://github.com/huggingface/parler-tts.git

In [ ]:
# 2. Voice Generation (Parler-TTS)
import torch
import soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Loading Parler-TTS onto {device}...")

model = ParlerTTSForConditionalGeneration.from_pretrained("parler-tts/parler-tts-mini-v1").to(device)
tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler-tts-mini-v1")

# Check for custom voice prompt
voice_prompt_path = INPUT_DIR / 'voice_prompt.txt'
if voice_prompt_path.exists():
    voice_prompt = voice_prompt_path.read_text().strip()
    print(f"\nUsing custom voice style: '{voice_prompt}'")
else:
    voice_prompt = "A professional male narrator speaking in a clear, deep, and dramatic voice. He speaks slowly and deliberately."
    print(f"\nNo voice_prompt.txt found. Using default voice style:\n'{voice_prompt}'")

for txt_file in INPUT_DIR.glob('*.txt'):
    if txt_file.name == 'voice_prompt.txt':
        continue
        
    text = txt_file.read_text().strip()
    if not text:
        continue
        
    out_wav = AUDIO_DIR / f"{txt_file.stem}.wav"
    print(f"\nGenerating audio for {txt_file.name}...")
    
    # Parler-TTS inputs
    input_ids = tokenizer(voice_prompt, return_tensors="pt").input_ids.to(device)
    prompt_input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    
    # Generate
    generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
    audio_arr = generation.cpu().numpy().squeeze()
    
    sf.write(str(out_wav), audio_arr, model.config.sampling_rate)
    print(f"Audio saved to {out_wav}")


In [ ]:
# 3. Captioning (Faster-Whisper)
from faster_whisper import WhisperModel
import json

print('Loading Whisper...')
model = WhisperModel('small', device='cuda', compute_type='float16')

for wav_file in AUDIO_DIR.glob('*.wav'):
    print(f'Captioning {wav_file.name}...')
    segments, info = model.transcribe(str(wav_file), word_timestamps=True)
    
    cap_data = {'audio_file': str(wav_file), 'segments': []}
    for i, seg in enumerate(segments):
        words = [{'word': w.word, 'start': w.start, 'end': w.end} for w in seg.words]
        cap_data['segments'].append({
            'id': i, 
            'text': seg.text.strip(), 
            'start': seg.start, 
            'end': seg.end, 
            'words': words
        })
        
    out_cap = CAPTIONS_DIR / f"{wav_file.stem}.json"
    with open(out_cap, 'w') as f:
        json.dump(cap_data, f, indent=2)
    print(f"Captions saved to {out_cap}")

In [ ]:
# 4. Matching
import re

clip_index_path = INPUT_DIR / 'clip_index.json'
clips = []
if clip_index_path.exists():
    clips = json.loads(clip_index_path.read_text()).get('clips', [])
    print(f"Loaded {len(clips)} clips from clip_index.json")
else:
    print("No clip_index.json found. All segments will use AI images.")

def extract_keywords(text):
    words = re.sub(r'[^a-z0-9\s]', '', text.lower()).split()
    stops = {'the', 'is', 'at', 'which', 'on', 'in', 'a', 'an', 'of', 'to', 'and', 'or', 'for'}
    return {w for w in words if w not in stops and len(w) > 1}

for cap_file in CAPTIONS_DIR.glob('*.json'):
    cap_data = json.loads(cap_file.read_text())
    manifest = {'audio_file': cap_data['audio_file'], 'segments': []}
    
    for seg in cap_data['segments']:
        best_clip = None
        best_score = 0
        seg_kw = extract_keywords(seg['text'])
        
        for c in clips:
            clip_kw = extract_keywords(c.get('action', '')) | set(c.get('tags', []))
            score = len(seg_kw & clip_kw)
            if score > best_score:
                best_score = score
                best_clip = c
        
        entry = {
            'id': seg['id'], 
            'text': seg['text'], 
            'start': seg['start'], 
            'end': seg['end'], 
            'words': seg['words']
        }
        
        if best_clip and best_score > 0:
            entry['visual_type'] = 'clip'
            entry['visual_source'] = best_clip['filename']
            entry['clip_start'] = 0.0
        else:
            entry['visual_type'] = 'ai_image'
            clean_text = seg['text'][:100].replace('"', '').replace("'", '')
            entry['visual_source'] = f"Cinematic still depicting: {clean_text}. Dramatic lighting, high detail, 9:16 vertical composition."
            
        manifest['segments'].append(entry)
    
    out_manifest = OUTPUT_DIR / f"manifest_{cap_file.stem}.json"
    with open(out_manifest, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest saved to {out_manifest}")

In [ ]:
# 5. Image Generation (SDXL)
import torch
from diffusers import DiffusionPipeline

print('Loading Stable Diffusion XL...')
pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', 
    torch_dtype=torch.float16, 
    use_safetensors=True, 
    variant='fp16'
)
pipe.to('cuda')

for man_file in OUTPUT_DIR.glob('manifest_*.json'):
    man_data = json.loads(man_file.read_text())
    for seg in man_data['segments']:
        if seg.get('visual_type') == 'ai_image':
            prompt = seg['visual_source']
            out_img = IMAGES_DIR / f"seg_{seg['id']:04d}.png"
            
            if not out_img.exists():
                print(f'Generating image for segment {seg["id"]}...')
                image = pipe(
                    prompt=prompt, 
                    width=1080, 
                    height=1920, 
                    num_inference_steps=25
                ).images[0]
                image.save(out_img)
            else:
                print(f'Image {out_img.name} already exists. Skipping.')

In [ ]:
# 6. Package and Zip Outputs
import shutil

print('Zipping outputs...')
# Zip the whole content directory except the inputs and huge repos to save time
os.makedirs('/content/zip_staging', exist_ok=True)
for folder in ['audio', 'captions', 'images', 'output']:
    src = f'/content/{folder}'
    dst = f'/content/zip_staging/{folder}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

shutil.make_archive('/content/explainer_outputs', 'zip', '/content/zip_staging')
print('\n\u2705 Done! Please download /content/explainer_outputs.zip')

# Optional: Automatically trigger download in browser
try:
    from google.colab import files
    files.download('/content/explainer_outputs.zip')
except ImportError:
    pass